# 03 — Exploratory Data Analysis

Loads `data/processed/features.csv` and explores distributions, class balance, and feature relationships.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

FIGURES_DIR = Path.cwd().parent / "outputs" / "figures"

df = pd.read_csv(Path.cwd().parent / "data" / "processed" / "features.csv")
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

## Win/Loss Tally by Current Win Streak

Streaks beyond ±10 have very few observations and are clipped for readability.

In [ ]:
CLIP = 10

plot_df = df.copy()
plot_df["streak_clipped"] = plot_df["win_streak"].clip(-CLIP, CLIP)

tally = (
    plot_df.groupby(["streak_clipped", "target"])
    .size()
    .reset_index(name="count")
)
tally["outcome"] = tally["target"].map({1: "Win", 0: "Loss"})

fig, ax = plt.subplots(figsize=(14, 5))

sns.barplot(
    data=tally,
    x="streak_clipped",
    y="count",
    hue="outcome",
    palette={"Win": "#2ecc71", "Loss": "#e74c3c"},
    ax=ax,
)

ax.set_title("Win / Loss Tally by Current Win Streak (going into game)", fontsize=14)
ax.set_xlabel("Win streak going into game  (negative = loss streak)")
ax.set_ylabel("Number of games")
ax.legend(title="Outcome")
ax.axvline(x=ax.get_xticks()[CLIP], color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_win_streak_tally.png", dpi=150)
plt.show()